In [ ]:
from pathlib import Path
import cv2
import numpy as np



In [ ]:
#This cell is used convert PNG masks into YOLO format

masks_dir = Path("datasets/segmentation/optic_disk/train/masks")
labels_dir = Path("datasets/segmentation/optic_disk/train/labels")

labels_dir.mkdir(parents=True, exist_ok=True)

CLASS_ID = 0

for mask_path in masks_dir.glob("*.png"):

    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

    if mask is None:
        print(f"Unable to read: {mask_path}")
        continue

    h, w = mask.shape

    # Binarization
    _, mask_bin = cv2.threshold(
        mask, 127, 255, cv2.THRESH_BINARY
    )

    # Extract external contours
    contours, _ = cv2.findContours(
        mask_bin,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    label_path = labels_dir / f"{mask_path.stem}.txt"

    if not contours:
        print(f"No contours found: {mask_path.name}")
        label_path.write_text("")
        continue

    # Only one optic disc per image
    contour = max(contours, key=cv2.contourArea)

    # Convert contour to an array of points (N, 2)
    contour = contour.reshape(-1, 2)

    if len(contour) < 3:
        print(f"Invalid contour: {mask_path.name}")
        label_path.write_text("")
        continue

    # Normalize coordinates
    normalized = contour.astype(np.float64)
    normalized[:, 0] /= w
    normalized[:, 1] /= h

    # YOLO format: class_id x1 y1 x2 y2 ...
    coordinates = " ".join(
        f"{value:.6f}"
        for value in normalized.flatten()
    )

    label_path.write_text(
        f"{CLASS_ID} {coordinates}\n"
    )

print("Conversion completed.")